In [ ]:
import SimpleITK as sitk
import numpy as np 
import os
import glob
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output
from tqdm import tqdm
import itertools
import pydicom as pdm
import tensorflow as tf

import pandas as pd
for gpu in tf.config.experimental.list_physical_devices('GPU'):
    tf.config.experimental.set_virtual_device_configuration(gpu, [tf.config.experimental.VirtualDeviceConfiguration(memory_limit=6144)]) # 4096 change back to this after param search
    print('\n\n{}\n\n'.format(gpu))
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Conv3D, MaxPooling3D, Dropout, concatenate, Conv3DTranspose, LeakyReLU, BatchNormalization
from tensorflow.keras.optimizers import *
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, TensorBoard, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix

# LOAD DATA

### Create NIFTI Files

In [ ]:
from tqdm import tqdm
import itertools
import pydicom as pdm

def proc_dicoms(subject_dir):
    """Separate sequences in dicom directory and export to nifti files"""


    # Create nifti output directory if none exists

    nifti_dir = os.path.join(subject_dir, 'image_files_3D')

    if not os.path.exists(nifti_dir):
        os.mkdir(nifti_dir)


    # Create metadata output directory if none exists

    metadata_dir = os.path.join(subject_dir, 'metadata')

    if not os.path.exists(metadata_dir):
        os.mkdir(metadata_dir)


    # Instantiate image and metadata reader objects

    img_reader = sitk.ImageSeriesReader()
    metadata_reader = sitk.ImageFileReader()


    # Get unique series IDs in directory

    subdir_list = glob.glob(os.path.join(subject_dir, '**'), recursive=True)  # Get all possible sub-paths

    series_ID_list = [img_reader.GetGDCMSeriesIDs(i) for i in subdir_list]  # Search sub-paths for series'

    series_ID_list = np.unique(list(itertools.chain(*series_ID_list)))  # flatten list and keep only unique series IDs


    # Load each series and export to nifti file

    file_list = []  # track nifti files saved
    img_size_list = []  # track dimensions of each image volume


    for i, series_ID in enumerate(tqdm(series_ID_list)):

        dicom_files = img_reader.GetGDCMSeriesFileNames(subject_dir, series_ID, recursive=True)


        # Get metadata

        metadata = get_metadata(dicom_files)

        protocol_name = str(metadata['ProtocolName'][0])


        try:

            # Load image volume

            img_reader.SetFileNames(dicom_files)
            img = img_reader.Execute()

            img_size = img.GetSize()
            img_size_list.append(img_size)


        except:

            # Some dicoms have erroneous dimensions; try removing and reloading image volume

            print('Erroneous DICOM dims detected')

            # Get dimensions of each dicom in series

            dim_list = []

            for dicom in dicom_files:

                metadata_reader.SetFileName(dicom)
                metadata_reader.ReadImageInformation()
                rows, cols = metadata_reader.GetMetaData('0028|0010'), metadata_reader.GetMetaData('0028|0011')

                dim_list.append((rows, cols))


            # Remove dicoms with outlier dims

            mode_dims = max(dim_list, key=dim_list.count)
            idx = [i for i, dims in enumerate(dim_list) if dims == mode_dims]
            dicom_files = np.asarray(dicom_files)[idx]

            # Reload dicom series

            img_reader.SetFileNames(dicom_files)
            img = img_reader.Execute()


        # Export to nifti file

        save_name = protocol_name.replace(' ', '_').replace('/', '_') + f'_{i}'

        sitk.WriteImage(img, os.path.join(nifti_dir, save_name + '.nii'))

        file_list.append(save_name)


        # Save metadata

        np.save(os.path.join(metadata_dir, save_name + '_metadata.npy'), metadata)


    # Save log

    log_file = {subject_dir: list(zip(file_list, series_ID_list, img_size_list))}

    with open(os.path.join(nifti_dir, 'nifti_creation_log.txt'), 'w') as file:

        for key, value in log_file.items():
            file.write(f'{key}:\n\n')

            for sub_value in value:
                file.write(f'{sub_value}\n')


In [ ]:
def get_metadata(dcm_path_list):
    """
    Get relevant metadata fields from dicoms

    INPUTS:
    dcm_path_list: list of all dicom paths belonging to MRI sequence

    OUTPUTS:
    metadata: dictionary of metadata fields
    """

    metadata = {'ProtocolName': [],
                'SeriesInstanceUID': [],
                'AcquisitionMatrix': [],
                'VoxelSize': [],
                'FlipAngle': [],
                'RepetitionTime': [],
                'EchoTime': [],
                'ImageOrientationPatient': [],
                'ImagePositionPatient': [],
                'TransmitCoilName': [],
                'Manufacturer': [],
                'ManufacturerModelName': []}

    # Get fields of interest from all dicoms in sequence
    for dcm_file in dcm_path_list:

        dcm = pdm.dcmread(dcm_file)

        for key_name in metadata.keys():

            try:
                if key_name == 'VoxelSize':
                    metadata[key_name].append([dcm.PixelSpacing[0], dcm.PixelSpacing[1], dcm.SliceThickness])
                else:
                    metadata[key_name].append(dcm[key_name].value)
            except:
                metadata[key_name].append('UNKNOWN')

    # Keep only unique entries
    for key_name in metadata.keys():

        if key_name == 'ImagePositionPatient':
            metadata[key_name] = np.array(metadata[key_name]).max(axis=0).flatten()
        else:
            metadata[key_name] = np.unique(metadata[key_name], axis=0).flatten()

    return metadata

In [ ]:
dir_dess = "Practice Dicoms/DESS_024/"
dess_var = "DESS"

dir_ciss = "Practice Dicoms/CISS_024/"
ciss_var = "CISS"

In [ ]:
# Create NIFTI Files

def nii_file(top_dir, search_var):
    nifti_path_list = glob.glob(os.path.join(top_dir, f'image_files_3D/*{search_var}*.nii'))

    if len(nifti_path_list) == 0:
        
        print('Creating NIFTI and metadata files')
        
        proc_dicoms(top_dir)
        
        nifti_path_list = glob.glob(os.path.join(top_dir, f'image_files_3D/*{search_var}*.nii'))

    if len(nifti_path_list) == 1:
        print('Existing target sequence NIFTI file found; skipping NIFTI creation step')

        path_num = 0

    elif len(nifti_path_list) > 1:

        print('WARNING: multiple target sequences detected, choose which one to use: \n')

        [print(i, ')', path, '\n') for (i, path) in enumerate(nifti_path_list)]

        path_num = int(input())

    elif len(nifti_path_list) == 0:
        print('No matching sequence found')

    metadata_path = glob.glob(os.path.join(top_dir, f'metadata/*{search_var}*.npy'))[path_num]
    nii_file_path = nifti_path_list[path_num]
    
    return nii_file_path, metadata_path


In [ ]:
# Make DESS and CISS NIFTI Files 

dess_nii_path, dess_metadata_path = nii_file(dir_dess, dess_var)
ciss_nii_path, ciss_metadata_path = nii_file(dir_ciss, ciss_var)

### Set up Image and Metadata

In [ ]:
def get_image(nii_path, metadata_path):
    img_sitk = sitk.ReadImage(nii_path)
    img = sitk.GetArrayFromImage(img_sitk)
    metadata = np.load(metadata_path)
    
    num_echoes = len(metadata['EchoTime'])

    if num_echoes > 1:
        img = np.stack(np.split(img, num_echoes), axis=-1) # split echoes if multiecho sequence
        
    return img

## DEFINE Crop Image

In [ ]:
def crop_image(img, crop_range = [20, 256, 256]):
    img_dims = img.shape
    #Slices
    slice_start,slice_stop = img_dims[0]//2 - crop_range[0]//2, img_dims[0]//2 + crop_range[0]//2

    # Row
    row_start,row_stop = img_dims[1]//2 - crop_range[1]//2, img_dims[1]//2 + crop_range[1]//2

    # Columns 
    col_start,col_stop = img_dims[2]//2 - crop_range[2]//2, img_dims[2]//2 + crop_range[2]//2
    
    img_cropped = img[slice_start:slice_stop, row_start:row_stop, col_start:col_stop, ...]

    return img_cropped


## DEFINE Jitter and Normalize

In [ ]:
def random_jitter(image):
    image = tf.image.random_flip_left_right(image)
    return image

In [ ]:
def normalize_image(image):
    img_processed = image.astype(Float)
    img_processed = (img_processed / 127.5) - 1.0 # Normalize to [-1,1]
    img_processed = img_processed[..., np.newaxis] # Add channel Axis
    
    return img_processed

# Define Preprocess for train and test images

In [ ]:
def preprocess_image_train(image):
    image = crop_image(image)
    image = random_jitter(image)
    image = normalize_image(image)
    
    return image    

In [ ]:
def preprocess_image_test(image):
    image = crop_image(image)
    image = normalize_image(image)
    
    return image    

### TO DO LOAD DATA, BATCH DATA, and SHUFFLE (training only)

### Conv Block

In [ ]:
def conv_block(input_, filter_size, kernel_size, alpha, batch_normalization):
    initializer = tf.random_normal_initializer(mean = 0., stddev = 0.02)
    conv = Conv3D(filter_size, kernel_size=kernel_size, padding='same',kernel_initializer=initializer)(input_)
    
    if batch_normalization:
        conv = BatchNormalization()(conv)
    conv = LeakyReLU(alpha=alpha)(conv)
    conv = Conv3D(filter_size, kernel_size=kernel_size, padding='same', kernel_initializer='he_normal')(conv)
    if batch_normalization:
        conv = BatchNormalization()(conv)
    conv = LeakyReLU(alpha=alpha)(conv)

    return conv

### Downsample Block

In [ ]:
def down_block(input_, filter_size, kernel_size,  norm_type='instancenorm', apply_norm = True):
    """
    3D Downsampling Block using conv_block and MaxPooling3D.

    Args:
        input_ (tensor): Input tensor.
        filter_size (int): Number of filters.
        kernel_size (int or tuple): Size of the convolution kernel.
        alpha (float): LeakyReLU negative slope.
        dropout (float): Dropout rate.
        batch_normalization (bool): Whether to apply batch normalization.

    Returns:
        downsampled tensor (tensor), skip connection tensor (tensor)
    """
    initializer = tf.random_normal_initializer(0., 0.02)
    
    result = tf.keras.Sequential()
    result.add(tf.keras.layers.Conv3D(input_, filter_size, kernel_size, strides=2, padding='same',
                            kernel_initializer=initializer, use_bias=False))
    
    if apply_norm:
        if norm_type == 'batchnorm':
            result.add(tf.keras.layers.BatchNormalization())
        elif norm_type == 'instancenorm':
            result.add(tf.layers.InstanceNormalization())

    result.add(tf.keras.layers.LeakyReLU())

    return result

### Upsample Block

In [ ]:
def up_block(input_, skip_connection, filter_size, kernel_size,  norm_type='instancenorm', apply_dropout = False):
    """
    3D Upsampling Block using strided Conv3D instead of Conv3DTranspose.

    Args:
        input_ (tensor): Input tensor.
        skip_connection (tensor): Corresponding skip connection tensor.
        filter_size (int): Number of filters.
        kernel_size (int or tuple): Size of the convolution kernel.
        alpha (float): LeakyReLU negative slope.
        batch_normalization (bool): Whether to apply batch normalization.

    Returns:
        Up-sampled tensor (tensor)
    """
    initializer = tf.random_normal_initializer(mean = 0., stddev=0.02)
    result = tf.keras.Sequential()
    result.add(tf.keras.layers.Conv3DTranspose(input_,filter_size, kernel_size, strides=2, padding='same',
                                    kernel_initializer=initializer, use_bias=False))
    
    if norm_type == 'batchnorm':
        result.add(tf.keras.layers.BatchNormalization())
    elif norm_type == 'instancenorm':
        result.add(tf.keras.layers.InstanceNormalization())

    if apply_dropout:
        result.add(tf.keras.layers.Dropout(0.5))

    result.add(tf.keras.layers.ReLU())

    return result

### U-Net Generator

In [ ]:
def unet_generator_3D(output_channels, norm_type='instancenorm'):
    """ 
    Args: 
        output_channels: Number of output channels (e.g., 1 for segmentation)
        norm_type: Type of normalization. Either 'batchnorm' or 'instancenorm'

    Returns:
        Generator Model (3D U-Net)
    """
    down_stack = [
        down_block(64, (4,4,4), norm_type, apply_norm=False),  # (bs, 10, 128, 128, 64)
        down_block(128, (4,4,4), norm_type),                   # (bs, 5, 64, 64, 128)
        down_block(256, (4,4,4), norm_type),                   # (bs, 3, 32, 32, 256)
        down_block(512, (4,4,4), norm_type),                   # (bs, 2, 16, 16, 512)
        down_block(512, (4,4,4), norm_type),                   # (bs, 1, 8, 8, 512)
    ]

    up_stack = [
        up_block(512, (4,4,4), norm_type, apply_dropout=True),  # (bs, 2, 16, 16, 512)
        up_block(256, (4,4,4), norm_type, apply_dropout=True),  # (bs, 3, 32, 32, 256)
        up_block(128, (4,4,4), norm_type),                      # (bs, 5, 64, 64, 128)
        up_block(64, (4,4,4), norm_type),                       # (bs, 10, 128, 128, 64)
    ]

    initializer = tf.random_normal_initializer(0., 0.02)
    last = tf.keras.layers.Conv3DTranspose(output_channels, (4,4,4), strides=2, padding='same',
                                kernel_initializer=initializer, activation='tanh') # (bs, 20, 256, 256, output_channels)

    concat = tf.keras.layers.Concatenate()

    inputs = tf.keras.layers.Input(shape=(20, 256, 256, 1))
    x = inputs

    # Downsampling
    skips = []
    for down in down_stack:
        x = down(x)
        skips.append(x)
    
    skips = reversed(skips[:-1])

    # Upsampling and skip connections
    for up, skip in zip(up_stack, skips):
        x = up(x)
        x = concat([x, skip])

    x = last(x)

    return tf.keras.Model(inputs=inputs, outputs=x)


### Discriminator

In [ ]:
def discriminator_3d(norm_type = 'instancenorm', target=True):
    """ 3D PatchGAN Discriminator Model 
    
    Args:
        norm_type: Type of normalization. Either 'batchnorm' or 'instancenorm'
        target: Bool, indicating whether target image is an input or not 

    Returns:
        Discriminator model 
    """
    
    initializer = tf.random_normal_initializer(0., 0.2)

    inp = tf.keras.layers.Input(shape=[None, None, None, 1], name='input_image')  # Input shape for 3D image
    x = inp

    if target: 
        tar = tf.keras.layers.Input(shape=[None, None, None, 1], name='target_image')  # Target image shape for 3D
        x = tf.keras.layers.concatenate([inp, tar])  # Concatenate input and target (bs, D, H, W, 2 channels)

    # Downsampling using 3D convolutions and 3D pooling
    down1 = down_block(64, 4, norm_type, False)(x)  # (bs, D/2, H/2, W/2, 64)
    down2 = down_block(128, 4, norm_type)(down1)   # (bs, D/4, H/4, W/4, 128)
    down3 = down_block(256, 4, norm_type)(down2)   # (bs, D/8, H/8, W/8, 256)

    # Padding and 3D convolution layer
    zero_pad1 = tf.keras.layers.ZeroPadding3D()(down3)  # (bs, D+1, H+1, W+1, 256)
    conv = tf.keras.layers.Conv3D(
        512, 4, strides=1, kernel_initializer=initializer, use_bias=False)(zero_pad1)  # (bs, D-2, H-2, W-2, 512)

    if norm_type.lower() == 'batchnorm':
        norm1 = tf.keras.layers.BatchNormalization()(conv)
    elif norm_type.lower() == 'instancenorm':
        norm1 = tf.keras.layers.InstanceNormalization()(conv)

    leaky_relu = tf.keras.layers.LeakyReLU()(norm1)

    # Padding and final convolution for PatchGAN output
    zero_pad2 = tf.keras.layers.ZeroPadding3D()(leaky_relu)  # (bs, D, H, W, 512)

    last = tf.keras.layers.Conv3D(
        1, 4, strides=1, kernel_initializer=initializer)(zero_pad2)  # (bs, D-3, H-3, W-3, 1)

    if target:
        return tf.keras.Model(inputs=[inp, tar], outputs=last)
    else:
        return tf.keras.Model(inputs=inp, outputs=last)


# APPLY Generator and Discriminator

In [ ]:
OUTPUT_CHANNELS = 1  # For black-and-white 3D images

generator_g = unet_generator_3D(OUTPUT_CHANNELS, norm_type='instancenorm')
generator_f = unet_generator_3D(OUTPUT_CHANNELS, norm_type='instancenorm')

discriminator_x = discriminator_3d(norm_type='instancenorm', target=False)  # For 3D
discriminator_y = discriminator_3d(norm_type='instancenorm', target=False)  # For 3D


### LOSS FUNCTIONS

In [ ]:
LAMBDA = 10

loss_obj = tf.keras.losses.BinaryCrossentropy(from_logits = True)

In [ ]:
def discriminator_loss(real, generated):
    real_loss = loss_obj(tf.ones_like(real), real)

    generated_loss = loss_obj(tf.zeros_like(generated), generated)

    total_disc_loss = real_loss + generated_loss 

    return total_disc_loss * 0.5

In [ ]:
def generator_loss(generated):
    return loss_obj(tf.ones_like(generated), generated)

In [ ]:
def calc_cycle_loss(real_image, cycled_image):
    loss1 = tf.reduce_mean(tf.abs(real_image - cycled_image))

    return LAMBDA * loss1

In [ ]:
def identity_loss(real_image, same_image):
    loss= tf.reduce_mean(tf.abs(real_image - same_image))
    return LAMBDA * 0.5 * loss
    

In [ ]:
generator_g_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
generator_f_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

discriminator_x_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_y_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

# CHECKPOINTS

In [ ]:
checkpoint_path = "./checkpoints/train"

ckpt = tf.train.Checkpoint(generator_g=generator_g,
                        generator_f=generator_f,
                        discriminator_x=discriminator_x,
                        discriminator_y=discriminator_y,
                        generator_g_optimizer=generator_g_optimizer,
                        generator_f_optimizer=generator_f_optimizer,
                        discriminator_x_optimizer=discriminator_x_optimizer,
                        discriminator_y_optimizer=discriminator_y_optimizer)

ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)

# if a checkpoint exists, restore the latest checkpoint.
if ckpt_manager.latest_checkpoint:
    ckpt.restore(ckpt_manager.latest_checkpoint)
    print ('Latest checkpoint restored!!')

# TRAINING

In [ ]:
EPOCHS = 10

In [ ]:
def generate_images(model, test_input):
    prediction = model(test_input)

    plt.figure(figsize(12,12))

    display_list = [test_input[0], prediction[0]]
    title = ['Input Image', 'Predicted Image']

    for i in range(2):
        plt.subplot(1,2,i+1)
        plt.title(title[i])
        plt.imshow(display_list[i]*0.5+0.5) # getting pixel values between [0,1] to plot it 
        plt.axis('off')
    plt.show()

In [ ]:
@tf.function

def train_step(real_x, real_y):
    with tf.GradientTape(persistent = True) as tape: #persistent is set to True because the tape is used more than once to calculate the gradients 

        # Generator G translates A -> B (X -> Y)
        # Generator F translates B -> A ( Y -> X)

        fake_y = generator_g(real_x, training = True)
        cycled_x = generator_f(fake_y, training = True)

        fake_x = generator_f(real_y, training = True)
        cycled_y = generator_g(fake_x, training = True)

        # same_x and same_y are used for identity loss 
        same_x = generator_f(real_x, training=True)
        same_y = generator_g(real_y, training = True)

        disc_real_x = discriminator_x(real_x, training = True)
        disc_real_y = discriminator_y(real_y, training = True)

        disc_fake_x = discriminator_x(fake_x, training=True)
        disc_fake_y = discriminator_y(fake_y, training = True)

        #calculate loss 
        gen_g_loss = generator_loss(disc_fake_y)
        gen_f_loss = generator_loss(disc_fake_x)

        total_cycle_loss = calc_cycle_loss(real_x, cycled_x) + calc_cycle_loss(real_y, cycled_y)

        # Total generator loss = adversarial loss + cycle loss
        total_gen_g_loss = gen_g_loss + total_cycle_loss + identity_loss(real_y, same_y)
        total_gen_f_loss = gen_f_loss + total_cycle_loss + identity_loss(real_x, same_x)

        disc_x_loss = discriminator_loss(disc_real_x, disc_fake_x)
        disc_y_loss = discriminator_loss(disc_real_y, disc_fake_y)

    # Calculate the gradients for generator and discriminators 
    generator_g_gradients = tape.gradient(total_gen_g_loss, generator_g.trainable_variables)
    generator_f_gradients = tape.gradient(total_gen_f_loss, generator_f.trainable_variables)

    discriminator_x_gradients = tape.gradient(disc_x_loss, discriminator_x.trainable_variables)
    discriminator_y_gradients = tape.gradient(disc_y_loss, discriminator_y.trainable_variables)

    # Apply the gradients to the optimizer
    generator_g_optimizer.apply_gradients(zip(generator_g_gradients, generator_g.trainable_variables))

    generator_f_optimizer.apply_gradients(zip(generator_f_gradients, generator_f.trainable_variables))

    discriminator_x_optimizer.apply_gradients(zip(discriminator_x_gradients, discriminator_x.trainable_variables))
    discriminator_y_optimizer.apply_gradients(zip(discriminator_y_gradients, discriminator_y.train))